# ViTextVQA — ablation đặc trưng OCR
Chọn kịch bản 1, 2, 3 hoặc 5 trong cell cấu hình. Kịch bản 4 là ABL1. Notebook dùng nhánh exp/vitextvqa-ocr-ablation, cấu hình ABL1 16×1, LR 1e-4, QA-ViT/Consformer/TWC tắt. Code này khác commit ABL1 lịch sử; cần chạy thêm kịch bản 4 trên cùng code để so sánh có kiểm soát.
Mỗi kịch bản có output và Hugging Face repo riêng. Chạy cell theo thứ tự.


## 1. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 2. Cấu hình — code (repo + commit) và module

In [ ]:
GIT_REPO       = "https://github.com/Kussssssss/ViSceT5.git"
GIT_COMMIT     = "exp/vitextvqa-ocr-ablation"
QAVIT          = "off"
OCR_SCENARIO = 1  # 1=text; 2=text+char; 3=text+char+bbox; 5=unaligned; 4=full control

# =========================================================================
# CHỌN 1 TRONG 4 CẤU HÌNH ABLATION (từ commit cc773b0: cả 4 cấu hình đều có đầy đủ Image + Question + OCR text + bbox + det + rec + char):
# 1. Baseline:                    USE_CONSFORMER = False, USE_TWC = False
# 2. Baseline + Consformer:       USE_CONSFORMER = True,  USE_TWC = False
# 3. Baseline + TWC:              USE_CONSFORMER = False, USE_TWC = True
# 4. Baseline + Consformer + TWC: USE_CONSFORMER = True,  USE_TWC = True (Full model)
# =========================================================================
USE_CONSFORMER = False
USE_TWC        = False
TWC_AUG_P      = 0.35
TWC_TAU        = 0.2
TWC_CONTRASTIVE = True    # True = Full · False = hai lượt + bản related, bỏ hoàn toàn TWC
LEARNING_RATE  = 1e-4   # phần pretrain sửa từ 3e-5 thành 1e-4 (new modules giữ 1e-4, scalar giữ 1e-3)
BATCH_SIZE     = 16
GRAD_ACCUM     = 1
DATASET        = "ViTextVQA"
PREDICT_CKPT   = "best"
PREDICT_MIN_DEV_EM = 0.0  # run đối chứng: 0.0 (điểm thấp cũng là kết quả cần có); run cải tiến: 31.0

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert QAVIT in ("off", "no_question", "question"), "QAVIT: off | no_question | question"
assert 0 <= TWC_AUG_P <= 1 and TWC_TAU > 0 and BATCH_SIZE >= 1 and GRAD_ACCUM >= 1 and LEARNING_RATE > 0
RUN_NAME = "-".join([m for m, on in (("qavit", QAVIT == "question"), ("qavitnq", QAVIT == "no_question"),
                                     ("consformer", USE_CONSFORMER),
                                     (f"twc-p{int(round(100 * TWC_AUG_P))}-tau{TWC_TAU:g}" if TWC_CONTRASTIVE else f"twoviews-p{int(round(100 * TWC_AUG_P))}", USE_TWC)) if on] or ["base"])
assert OCR_SCENARIO in (1, 2, 3, 4, 5)
RUN_NAME += f"-ocr{OCR_SCENARIO}-bs{BATCH_SIZE}x{GRAD_ACCUM}"
print(f"Code: {GIT_REPO} @ {GIT_COMMIT}")
print(f"Cấu hình: QA-ViT={QAVIT} | Consformer={USE_CONSFORMER} | TWC={USE_TWC}"
      f"{f' (p={TWC_AUG_P}, τ={TWC_TAU}, TWC_CONTRASTIVE={TWC_CONTRASTIVE})' if USE_TWC else ''} | lr={LEARNING_RATE:g} | batch={BATCH_SIZE}x{GRAD_ACCUM} | RUN_NAME={RUN_NAME}")

## 3. Lấy code đúng commit và đặt biến môi trường
Clone `GIT_REPO` (lần đầu), chuyển sang `GIT_COMMIT`, chạy `setup.sh` của commit đó, rồi đặt biến môi trường theo những gì `run_pipeline.py` của commit đó đọc được. Chạy lại cell này mỗi khi đổi cell 2.

In [ ]:
import os, re, subprocess, yaml
REPO_DIR = "/content/ViSceT5"
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q --branch {GIT_COMMIT} {GIT_REPO} {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin {GIT_REPO}
!git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags


def _resolve(ref):
    for cand in (f"origin/{ref}", ref):
        out = subprocess.run(["git", "rev-parse", "--verify", "-q", cand + "^{commit}"], capture_output=True, text=True)
        if out.returncode == 0:
            return out.stdout.strip()
    raise ValueError(f"Không tìm thấy commit/nhánh/tag '{ref}' trong {GIT_REPO}")


CODE_SHA = _resolve(GIT_COMMIT)
CODE_TAG = CODE_SHA[:7]
subprocess.run(["git", "checkout", "-q", "--detach", CODE_SHA], check=True)
!git log --oneline -1
!bash setup.sh

READ = set(re.findall(r'"([A-Z][A-Z_]+)": "[a-z_]+"', open("run_pipeline.py", encoding="utf-8").read()))
YAML_CFG = yaml.safe_load(open("configs/finetune.yaml", encoding="utf-8"))
for _name in ("ABLATION_USE_IMAGE", "ABLATION_USE_OCR", "ABLATION_USE_OCR_AUG", "ABLATION_USE_QACLIP", "CLIP_LATE_ADAPTER",
              "DATASET_NAME", "GRADIENT_ACCUMULATION_STEPS", "HF_REPO", "LEARNING_RATE", "MOCK_TEST", "MODEL_NAME_OR_PATH",
              "NEW_MODULE_LR", "NUM_TRAIN_EPOCHS", "OCR_AUG_MODE", "OCR_FEATURE_NORM", "OCR_RELATED", "OCR_LATTICE_KEEP",
              "OUTPUT_DIR", "PER_DEVICE_EVAL_BATCH_SIZE", "PER_DEVICE_TRAIN_BATCH_SIZE", "QAVIT_MODE",
              "RESUME_CHECKPOINT_ID", "RESUME_FROM_CHECKPOINT", "SCALAR_LR", "STAGE", "TRAIN_CONFIG",
              "TWC_ADV_PROBABILITY", "TWC_ANSWER_WEIGHT", "TWC_CONTRASTIVE", "TWC_FEATURE", "TWC_LOSS", "TWC_MASK_SAME_TEXT", "TWC_OBJECTIVE",
              "TWC_QUEUE_SIZE", "TWC_QUEUE_START", "TWC_RELATED_TRAIN", "TWC_TEMPERATURE", "TWC_VIEW", "TWC_WEIGHT",
              "USE_TWC", "GIT_BRANCH", "GIT_COMMIT", "PREDICT_CKPT_DIR", "PREDICT_DATASET", "PREDICT_SPLIT"):
    os.environ.pop(_name, None)

_b = lambda v: "true" if v else "false"
OUTPUT_DIR = f"/content/ViSceT5/output/{CODE_TAG}_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-{CODE_TAG}-{DATASET.lower()}-{RUN_NAME}"
ENV = {"DATASET_NAME": DATASET, "ABLATION_USE_OCR": _b(USE_CONSFORMER), "USE_TWC": _b(USE_TWC),
       "TWC_ADV_PROBABILITY": str(TWC_AUG_P), "TWC_TEMPERATURE": str(TWC_TAU)}
if USE_TWC and not TWC_CONTRASTIVE:
    ENV["TWC_CONTRASTIVE"] = "false"
_missing = [k for k in ENV if k not in READ]
assert not _missing, f"Commit {CODE_TAG}: run_pipeline.py không đọc {_missing} → không áp dụng được cấu hình ở cell 2."
if "QAVIT_MODE" in READ:
    ENV["QAVIT_MODE"] = QAVIT
else:
    assert QAVIT != "no_question", f"Commit {CODE_TAG} chưa có QA-ViT no_question: đặt QAVIT = \"off\" hoặc \"question\"."
if "ABLATION_USE_QACLIP" in READ:
    ENV["ABLATION_USE_QACLIP"] = _b(QAVIT == "question")
if "ABLATION_USE_OCR_AUG" in READ:
    ENV["ABLATION_USE_OCR_AUG"] = _b(USE_TWC)
if "LEARNING_RATE" in READ and "LEARNING_RATE" in globals() and LEARNING_RATE is not None:
    ENV["LEARNING_RATE"] = str(LEARNING_RATE)
for _name, _key, _value in (("PER_DEVICE_TRAIN_BATCH_SIZE", "per_device_train_batch_size", BATCH_SIZE),
                            ("GRADIENT_ACCUMULATION_STEPS", "gradient_accumulation_steps", GRAD_ACCUM)):
    if _name in READ:
        ENV[_name] = str(_value)
    else:
        assert YAML_CFG.get(_key) == _value, (f"Commit {CODE_TAG} không đọc {_name}: batch lấy từ yaml "
                                              f"({_key} = {YAML_CFG.get(_key)}) → đặt BATCH_SIZE / GRAD_ACCUM đúng giá trị đó.")
ENV.update({
    "OCR_USE_CHAR": _b(OCR_SCENARIO != 1),
    "OCR_USE_BBOX": _b(OCR_SCENARIO in (3, 4, 5)),
    "OCR_USE_VISUAL": _b(OCR_SCENARIO in (4, 5)),
    "OCR_ALIGN_SPATIAL_VISUAL": _b(OCR_SCENARIO != 5),
})
assert all(k in READ for k in ENV), "Commit does not expose every requested flag"
os.environ.update(ENV, OUTPUT_DIR=OUTPUT_DIR, HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(f"Code {CODE_TAG} | biến môi trường: {ENV}")
print(f"OUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}\nPREDICT_CKPT={PREDICT_CKPT}")

## 4. Kiểm tra trước khi train
Cell đầu chạy unit test và dừng nếu có test lỗi. Cell sau là mock test `--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước; không push, không resume từ HF.


In [ ]:
!git checkout -q --detach {CODE_SHA}
!pip install -q pytest
_tests = subprocess.run(["python", "-m", "pytest", "tests/test_vqa_pipeline.py", "-q", "--tb=short"], capture_output=True, text=True)
print(_tests.stdout[-6000:], _tests.stderr[-1000:])
assert _tests.returncode == 0, "Có unit test lỗi: gửi output ở trên trước khi train."


In [ ]:
!git checkout -q --detach {CODE_SHA}
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock MOCK_TEST=true STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5. Finetune (từ đầu, không pretrain) — log train có `twc_loss`, `twc_acc`, `twc_tau`, `view_loss` khi bật TWC

In [ ]:
!git checkout -q --detach {CODE_SHA}
!MOCK_TEST=false STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5b. (Khi cần) Đẩy checkpoint local lên HF rồi train tiếp
Dùng khi đã train mà **chưa có `HF_TOKEN`** (checkpoint chỉ nằm trên Colab): chạy cell 1 (token) → cell 2 (cấu hình, đúng commit và công tắc của lần train) → cell 3 → cell này. Cell đẩy các `checkpoint-N` trong `OUTPUT_DIR` (kèm optimizer/scheduler để resume) lên `HF_REPO`, sau đó chạy lại **cell 5**: `finetune.py` tự resume từ checkpoint mới nhất trên HF và từ đó push mỗi epoch. Chỉ checkpoint đã lưu cuối epoch mới có; phần đang train dở của epoch sau bị mất. Nếu runtime Colab đã reset thì không còn gì để đẩy.

In [ ]:
!ls -d {OUTPUT_DIR}/checkpoint-* 2>/dev/null
!python scripts/hf_io.py upload_checkpoints --repo {HF_REPO} --dir {OUTPUT_DIR}

## 6. Dự đoán dev + test (chỉ khi dev EM ở epoch 4 hoặc 5 đạt `PREDICT_MIN_DEV_EM` ở cell 2) (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 2. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng commit, công tắc của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
# Dự đoán trong Python mới để không dùng module Hugging Face đã cache trong kernel.
import json, os, subprocess, sys
_prediction_env = os.environ.copy()
_prediction_env["VISCET5_PREDICTION_CONTEXT"] = json.dumps({
    key: globals()[key] for key in ("CODE_SHA", "OUTPUT_DIR", "DATASET", "HF_REPO",
                                     "PREDICT_CKPT", "PREDICT_MIN_DEV_EM")
})
_bootstrap = """
import base64, json, os
from IPython.core.interactiveshell import InteractiveShell
shell = InteractiveShell.instance()
shell.user_ns.update(json.loads(os.environ['VISCET5_PREDICTION_CONTEXT']))
source = base64.b64decode('aW1wb3J0IG9zLCBzaHV0aWwsIGdsb2IsIGpzb24KaW1wb3J0IHN5cywgc2l0ZSwgc3VicHJvY2VzcywgZ2xvYgpkZWYgX2Vuc3VyZV9lbnYoKToKICAgICIiInRyYW5zZm9ybWVycyAvIGh1Z2dpbmdmYWNlX2h1YiAvIGV2YWx1YXRlIC8gYmVydF9zY29yZSBwaOG6o2kga2jhu5twIG5oYXUgKGPDoGkgZOG7nyBzYXUgc2V0dXAuc2gg4oaSIGzhu5dpIGltcG9ydCBraGkgY2jhuqVtIEJFUlRTY29yZSkuIiIiCiAgICBwcm9iZSA9ICJmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgSGZBcGksIGhmX2h1Yl9kb3dubG9hZDsgaW1wb3J0IHRyYW5zZm9ybWVycywgZXZhbHVhdGUsIGJlcnRfc2NvcmUsIHB5Y29jb2V2YWxjYXAiCiAgICBpZiBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItYyIsIHByb2JlXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSkucmV0dXJuY29kZSA9PSAwOgogICAgICAgIHJldHVybgogICAgcHJpbnQoIuKaoO+4jyBUaMawIHZp4buHbiBjaMawYSBraOG7m3AgKHRyYW5zZm9ybWVycyAvIGh1Z2dpbmdmYWNlX2h1YiAvIGV2YWx1YXRlKSDihpIgY8OgaSBs4bqhaSDEkcO6bmcgYuG6o24gY+G7p2EgcmVxdWlyZW1lbnRzLnR4dCIpCiAgICBmb3Igcm9vdCBpbiBzaXRlLmdldHNpdGVwYWNrYWdlcygpICsgW3NpdGUuZ2V0dXNlcnNpdGVwYWNrYWdlcygpXToKICAgICAgICBmb3IgZCBpbiBnbG9iLmdsb2Iob3MucGF0aC5qb2luKHJvb3QsICJ+KiIpKTogICMgdGjGsCBt4bulYyBwaXAgYuG7jyBk4bufOiB+cmFuc2Zvcm1lcnMgLi4uCiAgICAgICAgICAgIHNodXRpbC5ybXRyZWUoZCwgaWdub3JlX2Vycm9ycz1UcnVlKQogICAgc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLXEiLCAidHJhbnNmb3JtZXJzPT00LjQ1LjIiLCAiaHVnZ2luZ2ZhY2VfaHViPT0wLjI1LjIiLAogICAgICAgICAgICAgICAgICAgICJldmFsdWF0ZSIsICJiZXJ0LXNjb3JlIiwgImdpdCtodHRwczovL2dpdGh1Yi5jb20vc2FsYW5pei9weWNvY29ldmFsY2FwIl0sIGNoZWNrPVRydWUpCgpkZWYgX2d0X29rKGd0X3BhdGgsIHByZWRfY3N2KToKICAgICIiIkdUIHRlc3QgcGjhuqNpIGzDoCDEkcOhcCDDoW4gdGjhuq10IHbDoCDEkcO6bmcgdOG6rXAgxJHDoyBwcmVkaWN0OyBu4bq/dSBraMO0bmcgdGjDrCBraMO0bmcgY2jhuqVtICh0csOhbmggxJFp4buDbSAwIHbDtCBuZ2jEqWEpLiIiIgogICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgdHJ5OgogICAgICAgIGFubnMgPSBqc29uLmxvYWQob3BlbihndF9wYXRoLCBlbmNvZGluZz0idXRmLTgiKSlbImFubm90YXRpb25zIl0KICAgICAgICBndCA9IHtpbnQoYVsiaWQiXSk6IHN0cihhWyJhbnN3ZXJzIl1bMF0pLnN0cmlwKCkubG93ZXIoKSBmb3IgYSBpbiBhbm5zIGlmIGEuZ2V0KCJhbnN3ZXJzIil9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoIuKdjCBHVCB0ZXN0IGtow7RuZyDEkeG7jWMgxJHGsOG7o2M6IiwgZSkKICAgICAgICByZXR1cm4gRmFsc2UKICAgIGRpc3RpbmN0ID0gc2V0KGd0LnZhbHVlcygpKQogICAgaWYgInlvdXIgYW5zd2VyIiBpbiBkaXN0aW5jdCBvciAobGVuKGd0KSA+PSAxMDAgYW5kIGxlbihkaXN0aW5jdCkgPD0gNSk6CiAgICAgICAgcHJpbnQoZiLinYwgR1QgdGVzdCBsw6AgxJHDoXAgw6FuIGdp4bqjICh7bGVuKGRpc3RpbmN0KX0gZ2nDoSB0cuG7iyBraMOhYyBuaGF1KSDihpIga2jDtG5nIGNo4bqlbS4iKQogICAgICAgIHJldHVybiBGYWxzZQogICAgaWRzID0gc2V0KHBkLnJlYWRfY3N2KHByZWRfY3N2LCBrZWVwX2RlZmF1bHRfbmE9RmFsc2UpLmlsb2NbOiwgMF0uYXN0eXBlKGludCkpCiAgICBpZiBsZW4oaWRzICYgc2V0KGd0KSkgPCAwLjk5ICogbGVuKGlkcyk6CiAgICAgICAgcHJpbnQoZiLinYwgSUQgZOG7sSDEkW/DoW4gKHtsZW4oaWRzKX0pIGtow7RuZyBraOG7m3AgR1QgKHtsZW4oZ3QpfSk6IGNodW5nIHtsZW4oaWRzICYgc2V0KGd0KSl9IOKGkiBraMO0bmcgY2jhuqVtLiIpCiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBwcmludChmIuKchSBHVCB0ZXN0IGjhu6NwIGzhu4c6IHtsZW4oZ3QpfSBjw6J1LCB7bGVuKGRpc3RpbmN0KX0gxJHDoXAgw6FuIGtow6FjIG5oYXUsIGto4bubcCB7bGVuKGlkcyAmIHNldChndCkpfS97bGVuKGlkcyl9IElEIGThu7EgxJFvw6FuLiIpCiAgICByZXR1cm4gVHJ1ZQpfZW5zdXJlX2VudigpCmZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaSwgaGZfaHViX2Rvd25sb2FkCiFnaXQgY2hlY2tvdXQgLXEgLS1kZXRhY2gge0NPREVfU0hBfQohZ2l0IGZldGNoIC1xIG9yaWdpbiBleHAvdHdjLWJzNHgyLWFibGF0aW9uICYmIGdpdCBzaG93IEZFVENIX0hFQUQ6c2NyaXB0cy9oZl9pby5weSA+IC9jb250ZW50L2hmX2lvX25ldy5weQpfc3RhdGVzID0gc29ydGVkKGdsb2IuZ2xvYihmIntPVVRQVVRfRElSfS9jaGVja3BvaW50LSovdHJhaW5lcl9zdGF0ZS5qc29uIiksIGtleT1sYW1iZGEgcDogaW50KHAuc3BsaXQoImNoZWNrcG9pbnQtIilbLTFdLnNwbGl0KCIvIilbMF0pKQppZiBub3QgX3N0YXRlczoKICAgIF9yZW1vdGUgPSBzb3J0ZWQoKGYgZm9yIGYgaW4gSGZBcGkoKS5saXN0X3JlcG9fZmlsZXMoSEZfUkVQTykgaWYgZi5lbmRzd2l0aCgiL3RyYWluZXJfc3RhdGUuanNvbiIpKSwga2V5PWxhbWJkYSBwOiBpbnQocC5zcGxpdCgiY2hlY2twb2ludC0iKVstMV0uc3BsaXQoIi8iKVswXSkpCiAgICBfc3RhdGVzID0gW2hmX2h1Yl9kb3dubG9hZChIRl9SRVBPLCBfcmVtb3RlWy0xXSldIGlmIF9yZW1vdGUgZWxzZSBbXQpfZGV2cyA9IFsxMDAgKiBoWyJldmFsX2VtIl0gZm9yIGggaW4ganNvbi5sb2FkKG9wZW4oX3N0YXRlc1stMV0pKVsibG9nX2hpc3RvcnkiXSBpZiAiZXZhbF9lbSIgaW4gaF0gaWYgX3N0YXRlcyBlbHNlIFtdCnByaW50KCJEZXYgRU0gdGhlbyBlcG9jaDoiLCBbcm91bmQoZCwgMikgZm9yIGQgaW4gX2RldnNdKQppZiBtYXgoX2RldnNbMzo1XSwgZGVmYXVsdD0wKSA8IFBSRURJQ1RfTUlOX0RFVl9FTToKICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJEZXYgRU0g4bufIGVwb2NoIDQsIDUgY2jGsGEgxJHhuqF0IHtQUkVESUNUX01JTl9ERVZfRU19OiBi4buPIHF1YSBwcmVkaWN0LiIpClRFU1RfR1RfSlNPTiA9IGYiL2NvbnRlbnQvVmlTY2VUNS9ldmFsX2d0L3tEQVRBU0VUfV90ZXN0X2d0Lmpzb24iCmlmIG5vdCBvcy5wYXRoLmV4aXN0cyhURVNUX0dUX0pTT04pOgogICAgVEVTVF9HVF9KU09OID0gZiIvY29udGVudC97REFUQVNFVH1fdGVzdF9ndC5qc29uIgppZiBub3Qgb3MucGF0aC5leGlzdHMoVEVTVF9HVF9KU09OKToKICAgIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBmaWxlcwogICAgcHJpbnQoZiLirIbvuI8gQ2jhu41uIGZpbGUge29zLnBhdGguYmFzZW5hbWUoVEVTVF9HVF9KU09OKX0gxJHhu4MgdXBsb2FkIChi4bqlbSBDYW5jZWwgbuG6v3UgY2jGsGEgY8OzIOKGkiB24bqrbiBwcmVkaWN0LCBjaMawYSBjaOG6pW0gdGVzdCkiKQogICAgdXAgPSBmaWxlcy51cGxvYWQoKQogICAgaWYgdXA6CiAgICAgICAgc2h1dGlsLm1vdmUob3MucGF0aC5hYnNwYXRoKG5leHQoaXRlcih1cCkpKSwgVEVTVF9HVF9KU09OKQpwcmludCgiR1QgdGVzdDoiLCBURVNUX0dUX0pTT04gaWYgb3MucGF0aC5leGlzdHMoVEVTVF9HVF9KU09OKSBlbHNlICJjaMawYSBjw7MiKQoKaW1wb3J0IGpzb24KdGFnID0gIiIgaWYgREFUQVNFVCA9PSAiVmlUZXh0VlFBIiBlbHNlIGYie0RBVEFTRVR9XyIKIyBIRiDEkcOjIGPDsyBmaWxlIGThu7EgxJFvw6FuIGPhu6dhIGNoZWNrcG9pbnQgbsOgeSDihpIga2jDtG5nIHThuqNpIG1vZGVsIDEuNEdCLCBraMO0bmcgcHJlZGljdCBs4bqhaTogY2jhu4kgdOG6o2kgZmlsZSBk4buxIMSRb8OhbiB24buBLCBjaOG6pW0gdsOgIHVwIMSRaeG7g20uCmlmIG9zLnBhdGguZXhpc3RzKCIvY29udGVudC9wcmVkX2luZm8uanNvbiIpOgogICAgb3MucmVtb3ZlKCIvY29udGVudC9wcmVkX2luZm8uanNvbiIpCiFweXRob24gL2NvbnRlbnQvaGZfaW9fbmV3LnB5IGRvd25sb2FkX3ByZWRpY3Rpb25zIC0tcmVwbyB7SEZfUkVQT30gLS1kaXIge09VVFBVVF9ESVJ9IC0tY2twdCB7UFJFRElDVF9DS1BUfSAtLWluZm9fb3V0IC9jb250ZW50L3ByZWRfaW5mby5qc29uClBSRURfSU5GTyA9IGpzb24ubG9hZChvcGVuKCIvY29udGVudC9wcmVkX2luZm8uanNvbiIsIGVuY29kaW5nPSJ1dGYtOCIpKQpTQ09SRV9PTkxZID0gb3MucGF0aC5leGlzdHMob3MucGF0aC5qb2luKFBSRURfSU5GT1sicHJlZF9kaXIiXSwgZiJzdWJtaXNzaW9uX3t0YWd9dGVzdC5jc3YiKSkKaWYgU0NPUkVfT05MWToKICAgIHByaW50KCLinIUgSEYgxJHDoyBjw7MgZmlsZSBk4buxIMSRb8OhbiDihpIgYuG7jyBxdWEgdOG6o2kgbW9kZWwgdsOgIHByZWRpY3QsIGNo4buJIGNo4bqlbSDEkWnhu4NtIHLhu5NpIHVwIGvhur90IHF14bqjLiIpCiAgICBNT0RFTF9ESVIsIFBSRURfRElSLCBQUkVEX1JFUE9fUEFUSCA9IE5vbmUsIFBSRURfSU5GT1sicHJlZF9kaXIiXSwgUFJFRF9JTkZPWyJwcmVkX3BhdGhfaW5fcmVwbyJdCmVsc2U6CiAgICBpZiBvcy5wYXRoLmV4aXN0cygiL2NvbnRlbnQvY2twdF9pbmZvLmpzb24iKToKICAgICAgICBvcy5yZW1vdmUoIi9jb250ZW50L2NrcHRfaW5mby5qc29uIikKICAgICFweXRob24gc2NyaXB0cy9oZl9pby5weSBwcmVwYXJlIC0tcmVwbyB7SEZfUkVQT30gLS1kaXIge09VVFBVVF9ESVJ9IC0tY2twdCB7UFJFRElDVF9DS1BUfSAtLWluZm9fb3V0IC9jb250ZW50L2NrcHRfaW5mby5qc29uCiAgICBDS1BUX0lORk8gPSBqc29uLmxvYWQob3BlbigiL2NvbnRlbnQvY2twdF9pbmZvLmpzb24iLCBlbmNvZGluZz0idXRmLTgiKSkKICAgIE1PREVMX0RJUiwgUFJFRF9ESVIsIFBSRURfUkVQT19QQVRIID0gQ0tQVF9JTkZPWyJtb2RlbF9kaXIiXSwgQ0tQVF9JTkZPWyJwcmVkX2RpciJdLCBDS1BUX0lORk9bInByZWRfcGF0aF9pbl9yZXBvIl0KICAgIGFzc2VydCBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oTU9ERUxfRElSLCAibW9kZWwuc2FmZXRlbnNvcnMiKSksIGYiS2jDtG5nIGPDsyBtb2RlbCB0cm9uZyB7TU9ERUxfRElSfSIKCnByZWQgPSBvcy5wYXRoLmpvaW4oUFJFRF9ESVIsIGYic3VibWlzc2lvbl97dGFnfXRlc3QuY3N2IikKaWYgb3MucGF0aC5leGlzdHMocHJlZCk6CiAgICBwcmludChmIuKchSDEkMOjIGPDsyB7cHJlZH0g4oaSIGLhu48gcXVhIHByZWRpY3QsIGNo4buJIGNo4bqlbSDEkWnhu4NtLiIpCmVsc2U6CiAgICBmb3Igc3AgaW4gKCJkZXYiLCAidGVzdCIpOgogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKGYiL2NvbnRlbnQvVmlTY2VUNS9vdXRwdXQvc3VibWlzc2lvbl97dGFnfXtzcH0uY3N2Iik6CiAgICAgICAgICAgIG9zLnJlbW92ZShmIi9jb250ZW50L1ZpU2NlVDUvb3V0cHV0L3N1Ym1pc3Npb25fe3RhZ317c3B9LmNzdiIpCiAgICAhTU9DS19URVNUPWZhbHNlIFNUQUdFPXByZWRpY3QgUFJFRElDVF9TUExJVD1ib3RoIFBSRURJQ1RfREFUQVNFVD17REFUQVNFVH0gUFJFRElDVF9DS1BUX0RJUj17TU9ERUxfRElSfSBIRl9SRVBPPSBweXRob24zIHJ1bl9waXBlbGluZS5weSAyPiYxIHwgdGVlIHRyYWluX2V4ZWN1dGlvbi5sb2cKICAgIGZvciBzcCBpbiAoImRldiIsICJ0ZXN0Iik6CiAgICAgICAgc3JjID0gZiIvY29udGVudC9WaVNjZVQ1L291dHB1dC9zdWJtaXNzaW9uX3t0YWd9e3NwfS5jc3YiCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoc3JjKToKICAgICAgICAgICAgc2h1dGlsLmNvcHkyKHNyYywgb3MucGF0aC5qb2luKFBSRURfRElSLCBvcy5wYXRoLmJhc2VuYW1lKHNyYykpKQogICAgICAgICAgICBwcmludCgi4oaSIiwgb3MucGF0aC5qb2luKFBSRURfRElSLCBvcy5wYXRoLmJhc2VuYW1lKHNyYykpKQogICAgc2h1dGlsLmNvcHkyKCIvY29udGVudC9WaVNjZVQ1L3RyYWluX2V4ZWN1dGlvbi5sb2ciLCBvcy5wYXRoLmpvaW4oUFJFRF9ESVIsICJwcmVkaWN0X2xvZy50eHQiKSkKbG9nID0gb3MucGF0aC5qb2luKFBSRURfRElSLCAicHJlZGljdF9sb2cudHh0IikKZGV2X2xpbmVzID0gW2wuc3RyaXAoKSBmb3IgbCBpbiBvcGVuKGxvZywgZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJpZ25vcmUiKSBpZiAiW2Rldl0gRU09IiBpbiBsXSBpZiBvcy5wYXRoLmV4aXN0cyhsb2cpIGVsc2UgW10KaWYgZGV2X2xpbmVzOgogICAgZm9yIGxpbmUgaW4gZGV2X2xpbmVzWy0yOl06CiAgICAgICAgcHJpbnQobGluZSkKaWYgb3MucGF0aC5leGlzdHMoVEVTVF9HVF9KU09OKSBhbmQgb3MucGF0aC5leGlzdHMocHJlZCkgYW5kIF9ndF9vayhURVNUX0dUX0pTT04sIHByZWQpOgogICAgX2Vuc3VyZV9lbnYoKQogICAgIXB5dGhvbiAtbSB0cmFpbmluZy5iZW5jaG1hcmtfZXZhbCAtLWd0IHtURVNUX0dUX0pTT059IC0tcHJlZCB7cHJlZH0gLS1vdXQge1BSRURfRElSfS9ldmFsX3Rlc3QuanNvbgplbHNlOgogICAgcHJpbnQoIuKEue+4jyBDaMawYSDEkeG6t3QgVEVTVF9HVF9KU09OIOKGkiBjaMawYSBjaOG6pW0gdGVzdC4iKQoKaWYgb3MuZW52aXJvbi5nZXQoIkhGX1RPS0VOIikgYW5kIEhGX1JFUE86CiAgICAhcHl0aG9uIHNjcmlwdHMvaGZfaW8ucHkgdXBsb2FkX3ByZWRpY3Rpb25zIC0tcmVwbyB7SEZfUkVQT30gLS1kaXIge1BSRURfRElSfSAtLXBhdGhfaW5fcmVwbyB7UFJFRF9SRVBPX1BBVEh9IC0tZmlsZXMgc3VibWlzc2lvbl97dGFnfWRldi5jc3Ygc3VibWlzc2lvbl97dGFnfXRlc3QuY3N2IHByZWRpY3RfbG9nLnR4dCBldmFsX3Rlc3QuanNvbgplbHNlOgogICAgcHJpbnQoIuKaoO+4jyBLaMO0bmcgY8OzIEhGX1RPS0VOL0hGX1JFUE8g4oaSIGZpbGUgZOG7sSDEkW/DoW4gY2jhu4kgbuG6sW0gdHLDqm4gQ29sYWIsIGjDo3kgdOG6o2kgduG7gSB0csaw4bubYyBraGkgbmfhuq90IHBoacOqbi4iKQ==').decode('utf-8')
code = shell.transform_cell(source)
exec(compile(code, '<predict>', 'exec'), shell.user_ns)
"""
subprocess.run([sys.executable, "-u", "-c", _bootstrap], env=_prediction_env, check=True)
